In [1]:
from pathlib import Path
import pandas as pd


folder_path = Path("./mnist/test_results/9f56743a")

csv_files = list(folder_path.rglob("*.csv"))

for file in csv_files:
    print(file)
    columns = pd.read_csv(file, nrows=0).columns
    print(f"{file.name}: {len(columns)} columns")

mnist\test_results\9f56743a\dnn_cnn_dataset_9f56743a_automated_edge.csv
dnn_cnn_dataset_9f56743a_automated_edge.csv: 73 columns
mnist\test_results\9f56743a\mobilenet_v2_mnist_test_full_telemetry.csv
mobilenet_v2_mnist_test_full_telemetry.csv: 74 columns
mnist\test_results\9f56743a\moondream2_docci_instruct_mnist_test_predictions.csv
moondream2_docci_instruct_mnist_test_predictions.csv: 73 columns
mnist\test_results\9f56743a\qwen2vl_mnist_test_telemetry.csv
qwen2vl_mnist_test_telemetry.csv: 74 columns
mnist\test_results\9f56743a\yolo_test_dataset.csv
yolo_test_dataset.csv: 73 columns


In [2]:
from pathlib import Path
import pandas as pd


csv_files = sorted(folder_path.glob("*.csv"))

# First file = reference
reference_file = csv_files[0]
reference_columns = set(pd.read_csv(reference_file, nrows=0).columns)

print(f"Reference file: {reference_file.name}")
print(f"Reference column count: {len(reference_columns)}\n")

for file in csv_files[1:]:
    columns = set(pd.read_csv(file, nrows=0).columns)

    extra_columns = columns - reference_columns

    print(f"File: {file.name}")

    if extra_columns:
        print("Additional columns:")
        for col in sorted(extra_columns):
            print(f"  + {col}")
    else:
        print("No additional columns.")

    print("-" * 50)

Reference file: dnn_cnn_dataset_9f56743a_automated_edge.csv
Reference column count: 73

File: mobilenet_v2_mnist_test_full_telemetry.csv
No additional columns.
--------------------------------------------------
File: moondream2_docci_instruct_mnist_test_predictions.csv
No additional columns.
--------------------------------------------------
File: qwen2vl_mnist_test_telemetry.csv
No additional columns.
--------------------------------------------------
File: yolo_test_dataset.csv
No additional columns.
--------------------------------------------------


In [3]:
from pathlib import Path
import pandas as pd


csv_files = sorted(folder_path.glob("*.csv"))

if not csv_files:
    print("No CSV files found.")
else:
    reference_file = csv_files[0]
    reference_columns = list(pd.read_csv(reference_file, nrows=0).columns)

    print(f"Reference: {reference_file.name}")
    print(f"Reference column count: {len(reference_columns)}\n")

    all_mergeable = True

    for file in csv_files[1:]:
        columns = list(pd.read_csv(file, nrows=0).columns)

        missing = set(reference_columns) - set(columns)
        extra = set(columns) - set(reference_columns)

        print(f"{file.name}: {len(columns)} columns")

        if not missing and not extra:
            print("  ✓ Can be merged")
        else:
            all_mergeable = False
            print("  ✗ Column mismatch")

            if missing:
                print("  Missing columns:", sorted(missing))

            if extra:
                print("  Additional columns:", sorted(extra))

        print()

    if all_mergeable:
        print("✓ All CSV files can be merged.")
    else:
        print("✗ Some CSV files have different columns.")

Reference: dnn_cnn_dataset_9f56743a_automated_edge.csv
Reference column count: 73

mobilenet_v2_mnist_test_full_telemetry.csv: 73 columns
  ✓ Can be merged

moondream2_docci_instruct_mnist_test_predictions.csv: 73 columns
  ✓ Can be merged

qwen2vl_mnist_test_telemetry.csv: 73 columns
  ✓ Can be merged

yolo_test_dataset.csv: 73 columns
  ✓ Can be merged

✓ All CSV files can be merged.


In [4]:
dfs = [pd.read_csv(file) for file in csv_files]

merged_df = pd.concat(dfs, ignore_index=True)

merged_df.to_csv("merged.csv", index=False)

print("Merged rows:", len(merged_df))
print("Merged columns:", len(merged_df.columns))

Merged rows: 12
Merged columns: 73


In [ ]:
"""
Fixed FLOPS calculator for Qwen2-VL (Vision-Language Model)
Handles the multimodal nature of Qwen VLMs properly
"""

import torch
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from calflops import calculate_flops
from PIL import Image
import numpy as np

# Model configuration
MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def calculate_qwen2vl_flops_method1():
    """
    Method 1: Using Qwen2VLForConditionalGeneration with dummy image
    This is the correct approach for VLMs
    """
    print("=" * 60)
    print("Method 1: Qwen2VL with dummy image input")
    print("=" * 60)
    
    try:
        # Load model with correct class
        print(f"Loading model: {MODEL_ID}")
        model = Qwen2VLForConditionalGeneration.from_pretrained(
            MODEL_ID,
            trust_remote_code=True,
            device_map="auto",
            dtype=torch.float16,  # Use dtype instead of torch_dtype
        )
        
        # Load processor
        processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
        
        # Create dummy image (standard size: 448x448)
        dummy_image = Image.new('RGB', (448, 448), color='red')
        
        # Prepare input
        conversation = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                    },
                    {
                        "type": "text",
                        "text": "Describe this image."
                    }
                ],
            }
        ]
        
        text_prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
        inputs = processor(
            images=[dummy_image],
            text=text_prompt,
            padding=True,
            return_tensors="pt",
        )
        
        # Move to device
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        
        print("Calculating FLOPS...")
        flops, macs, params = calculate_flops(
            model=model,
            kwargs=inputs,
            include_backPropagation=False,
            output_as_string=True,
            output_precision=2,
        )
        
        print(f"\nResults:")
        print(f"  FLOPS: {flops}")
        print(f"  MACs:  {macs}")
        print(f"  Params: {params}")
        
        return flops, macs, params
        
    except Exception as e:
        print(f"Failed: {e}")
        return None, None, None


def calculate_qwen2vl_flops_method2(model_id=MODEL_ID):
    """
    Method 2: Manual FLOPS calculation for Qwen2-VL
    Based on model architecture
    """
    print("\n" + "=" * 60)
    print("Method 2: Manual calculation for Qwen2-VL")
    print("=" * 60)
    
    try:
        # Load model to get config
        print(f"Loading model: {model_id}")
        model = Qwen2VLForConditionalGeneration.from_pretrained(
            model_id,
            trust_remote_code=True,
            device_map="auto",
            dtype=torch.float16,
        )
        
        # Get model config
        config = model.config
        print(f"\nModel Architecture:")
        print(f"  Hidden size: {config.hidden_size}")
        print(f"  Number of layers: {config.num_hidden_layers}")
        print(f"  Number of attention heads: {config.num_attention_heads}")
        print(f"  Intermediate size (FFN): {config.intermediate_size}")
        
        # Count parameters
        total_params = sum(p.numel() for p in model.parameters())
        print(f"  Total parameters: {total_params:,}")
        
        # For VLMs, we need to consider:
        # 1. Vision encoder (ViT-like)
        # 2. Language model (Transformer)
        # 3. Projection layers
        
        # Typical sequence for Qwen2-VL:
        # - Image tokens: ~1024 (from patch embedding)
        # - Text tokens: ~128
        # - Total sequence length: ~1152
        
        seq_len_image = 1024  # Approximate image patches
        seq_len_text = 128    # Text tokens
        total_seq_len = seq_len_image + seq_len_text
        
        N = config.num_hidden_layers  # Number of transformer layers
        d = config.hidden_size         # Hidden dimension
        
        # Attention: 8*seq_len*d^2 + 4*seq_len^2*d
        attention_flops_per_layer = 8 * total_seq_len * (d ** 2) + 4 * (total_seq_len ** 2) * d
        
        # FFN: 8*seq_len*d^2 (approximate for MLP)
        ffn_flops_per_layer = 8 * total_seq_len * d * d
        
        # Total per layer
        flops_per_layer = attention_flops_per_layer + ffn_flops_per_layer
        
        # Total FLOPS (only for LLM part, not vision encoder)
        total_flops = N * flops_per_layer
        total_macs = total_flops // 2
        
        print(f"\nManual FLOPS Calculation (LLM component):")
        print(f"  Image tokens: ~{seq_len_image}")
        print(f"  Text tokens: {seq_len_text}")
        print(f"  Total sequence length: {total_seq_len}")
        print(f"  FLOPS per layer: {flops_per_layer:,.0f}")
        print(f"  Total FLOPS ({N} layers): {total_flops:,.0f}")
        print(f"  Total MACs: {total_macs:,.0f}")
        print(f"  FLOPS in billions: {total_flops / 1e9:.2f} B")
        print(f"\nNote: This calculation is for the LLM component only.")
        print(f"Vision encoder FLOPS would be additional.")
        
        return total_flops
        
    except Exception as e:
        print(f"Failed: {e}")
        return None


def calculate_qwen2vl_flops_method3():
    """
    Method 3: Using dummy tensors directly (faster, no image processing)
    """
    print("\n" + "=" * 60)
    print("Method 3: Direct tensor input (faster)")
    print("=" * 60)
    
    try:
        # Load model
        print(f"Loading model: {MODEL_ID}")
        model = Qwen2VLForConditionalGeneration.from_pretrained(
            MODEL_ID,
            trust_remote_code=True,
            device_map="auto",
            dtype=torch.float16,
        )
        
        # Create dummy inputs
        # For Qwen2-VL: input_ids and pixel_values
        batch_size = 1
        seq_len = 128  # Text sequence length
        
        dummy_input_ids = torch.randint(0, 32000, (batch_size, seq_len)).to(DEVICE)
        
        # Dummy pixel values (4D tensor: batch, channels, height, width)
        # Qwen2-VL typically uses 448x448 images
        dummy_pixel_values = torch.randn(
            batch_size, 3, 448, 448, 
            dtype=torch.float16, 
            device=DEVICE
        )
        
        # Prepare kwargs for calculate_flops
        kwargs = {
            "input_ids": dummy_input_ids,
            "pixel_values": dummy_pixel_values,
        }
        
        print("Calculating FLOPS...")
        flops, macs, params = calculate_flops(
            model=model,
            kwargs=kwargs,
            include_backPropagation=False,
            output_as_string=True,
            output_precision=2,
        )
        
        print(f"\nResults:")
        print(f"  FLOPS: {flops}")
        print(f"  MACs:  {macs}")
        print(f"  Params: {params}")
        
        return flops, macs, params
        
    except Exception as e:
        print(f"Failed: {e}")
        return None, None, None


def get_model_info():
    """
    Print detailed model information
    """
    print("\n" + "=" * 60)
    print("Model Information")
    print("=" * 60)
    
    try:
        model = Qwen2VLForConditionalGeneration.from_pretrained(
            MODEL_ID,
            trust_remote_code=True,
            device_map="auto",
            dtype=torch.float16,
        )
        
        print(f"\nModel: {MODEL_ID}")
        print(f"Model class: {model.__class__.__name__}")
        print(f"\nConfiguration:")
        print(f"  Model type: {model.config.model_type}")
        print(f"  Hidden size: {model.config.hidden_size}")
        print(f"  Number of layers: {model.config.num_hidden_layers}")
        print(f"  Number of attention heads: {model.config.num_attention_heads}")
        print(f"  KV heads: {model.config.num_key_value_heads}")
        print(f"  Intermediate size: {model.config.intermediate_size}")
        print(f"  Max position embeddings: {model.config.max_position_embeddings}")
        
        # Count parameters by component
        total_params = sum(p.numel() for p in model.parameters())
        print(f"\nParameters:")
        print(f"  Total: {total_params:,}")
        print(f"  Total (M): {total_params / 1e6:.2f}M")
        
    except Exception as e:
        print(f"Failed to load model: {e}")


if __name__ == "__main__":
    print(f"\n{'='*60}")
    print(f"Qwen2-VL FLOPS Calculator (Vision-Language Model)")
    print(f"{'='*60}")
    print(f"Model: {MODEL_ID}")
    print(f"Device: {DEVICE}")
    
    # Get model info first
    #get_model_info()
    
    # Try different methods
    print(f"\n{'='*60}")
    print("Calculating FLOPS using different methods...")
    print(f"{'='*60}")
    
    # Method 2: Manual calculation (usually works)
    flops2, macs2, params2 = calculate_qwen2vl_flops_method2()
    print(f"\nMethod 2 Results: FLOPS={flops2}, MACs={macs2}, Params={params2}")
    
    # Method 3: Direct tensor input
    #calculate_qwen2vl_flops_method3()
    
    # Method 1: With actual image (slower but more accurate)
    # calculate_qwen2vl_flops_method1()



Qwen2-VL FLOPS Calculator (Vision-Language Model)
Model: Qwen/Qwen2-VL-2B-Instruct
Device: cuda

Calculating FLOPS using different methods...

Method 2: Manual calculation for Qwen2-VL
Loading model: Qwen/Qwen2-VL-2B-Instruct


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


Model Architecture:
  Hidden size: 1536
  Number of layers: 28
  Number of attention heads: 12
  Intermediate size (FFN): 8960
  Total parameters: 2,208,985,600

Manual FLOPS Calculation (LLM component):
  Image tokens: ~1024
  Text tokens: 128
  Total sequence length: 1152
  FLOPS per layer: 51,640,270,848
  Total FLOPS (28 layers): 1,445,927,583,744
  Total MACs: 722,963,791,872
  FLOPS in billions: 1445.93 B

Note: This calculation is for the LLM component only.
Vision encoder FLOPS would be additional.

Method 2 Results: FLOPS=1445927583744, MACs=722963791872, Params=2208985600
